# Contexa — train and evaluate a context reranker

**Start here:** upload this notebook to Colab, select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**.
The default public pilot is self-contained and uses licensed, public, before-change source. No Contexa API token, paid model API, Hugging Face login, local setup, or private repository is needed. Original synthetic examples remain available as a smoke-test mode.

This fine-tunes a small pretrained **cross-encoder** to rank candidate code units against a developer's task. It does **not** train a general-purpose coding LLM, discover an entire repo by itself, generate code, or guarantee the exact context. Contexa must still retrieve candidates, resolve dependencies, remove secrets and enforce budgets.

**Important:** the public pilot uses patch-overlap weak labels. Changed units are positives; unchanged units are uncertain negatives. This cannot establish complete-context correctness. Both demo and pilot models remain `production_recommended: false`. Tests compare unseen repositories, not randomly split snippets. Public source can also occur in the pretrained model's unknown training corpus.

**Private data:** make a private copy if this notebook is shared with “anyone can edit.” Never upload production rows, credentials or unapproved customer source. The dataset scanner catches only some known leaks. No model or dataset is automatically published to Hugging Face. The ZIP omits raw training records and optimizer checkpoints; trained weights can still memorize sensitive data.

Save important checkpoints to Drive by enabling the option in Step 2. Colab's local `/content` files disappear when its VM resets; free GPUs and uninterrupted runtime are not guaranteed.


## 1. Install text-model dependencies
This uses Colab's existing PyTorch/GPU installation. It does not replace CUDA or install software on your computer.
If Colab requests a runtime restart after package installation, restart and run the notebook again.


In [ ]:
import os, sys, subprocess
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers==4.57.6", "safetensors==0.7.0", "tqdm==4.67.1"])
print("Dependencies ready. PyTorch/CUDA were not replaced.")


## 2. Configuration and GPU check
- The bundled public-change pilot runs by default, with no dataset upload needed.
- Set `USE_DEMO_DATA = True` for a synthetic smoke test, or disable both modes to upload reviewed labels.
- Set `PERSIST_TO_DRIVE = True` to keep epoch checkpoints through a VM reset. Google will ask you to authorize your own Drive mount. Rerunning with the same data/config resumes at the next epoch.
- If memory runs out, lower `BATCH_SIZE` to 2. Keep the same configuration when resuming.


In [ ]:
from pathlib import Path
import json, random, hashlib, time, math, gc, shutil
import torch
import importlib.metadata

USE_DEMO_DATA = False
USE_PUBLIC_DATA = True
PERSIST_TO_DRIVE = False
ALLOW_CPU = False  # Set True only if you intentionally accept much slower CPU training.
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
BASE_REVISION = "233902d25c440f23af6f7d6e94d2946bac0bee0a"
SEED = 42
EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LEARNING_RATE = 2e-5
MAX_LENGTH = 384
MAX_QUERY_TOKENS = 128
EVALUATION_BUDGET = 600  # Small enough to test token efficiency on the short demo units.
RESUME_IF_AVAILABLE = True

if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("No GPU is attached. Choose Runtime → Change runtime type → GPU, reconnect, and rerun. GPU availability depends on Colab.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content/drive/MyDrive/ContexaTraining")
else:
    WORK_ROOT = Path("/content/contexa_training")
    print("Checkpoints are temporary. Enable Drive persistence before long runs.")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)


## 3. Data preparation, split rules, and metrics
The following helpers run locally inside your Colab VM. They never execute candidate source code.


In [ ]:
"""Dependency-free data preparation and evaluation used inside the Colab notebook."""
import collections
import hashlib
import json
import math
import random
import re

FIELDS = ("task", "endpoints", "access", "rules", "response_shape")
LABELS = {"irrelevant": 0, "useful": 1, "essential": 2}


def query_text(row):
    return "\n".join(f"{key}: {row[key].strip()}" for key in FIELDS if row.get(key, "").strip())


def passage_text(row):
    return f"path: {row['path']}\nrole: {row.get('role', 'source')}\ncode:\n{row['content']}"


def make_demo_rows():
    """Original synthetic examples: smoke test only, not a production benchmark."""
    rows = []
    domains = ["Interview", "Invoice", "Booking", "Shipment", "Ticket", "Enrollment",
               "Appointment", "Expense", "Subscription", "Delivery", "Reservation", "Application"]
    for number, entity in enumerate(domains):
        plural = entity.lower() + "s"
        repo = f"demo-{number:02d}"
        stack = ["express-prisma", "django", "fastapi", "react"][number % 4]
        if stack == "express-prisma":
            units = [
                (f"src/controllers/{entity}Controller.ts", "controller", f"export async function create{entity}(req, res) {{ const result = await prisma.{entity.lower()}.create({{data: req.body}}); return sendSuccess(res, result, 201); }}"),
                (f"src/routes/{entity}Routes.ts", "route", f"router.post('/{plural}', requireRole('operator'), create{entity}); router.get('/{plural}', requireAuth, list{entity});"),
                ("prisma/schema.prisma", "model", f"model {entity} {{\n id Int @id @default(autoincrement())\n ownerId Int\n status {entity}Status\n @@map(\"{plural}\")\n}}\nenum {entity}Status {{\n PENDING\n ACTIVE\n}}"),
                ("src/middleware/auth.ts", "auth", f"// Access policy for {entity}\nexport const requireRole = (role) => (req,res,next) => req.user.role === role ? next() : sendError(res, 403, 'FORBIDDEN');"),
                ("src/utils/response.ts", "response", f"// Shared {entity} response contract\nexport const sendSuccess = (res, data, status=200) => res.status(status).json({{data}}); export const sendError = (res, status, code) => res.status(status).json({{error:{{code}}}});"),
                (f"src/services/list{entity}.ts", "service", f"export async function list{entity}(ownerId, page, limit) {{ const where={{ownerId}}; return {{data:await prisma.{entity.lower()}.findMany({{where,skip:(page-1)*limit,take:limit}}),paging:{{total:await prisma.{entity.lower()}.count({{where}}),page,limit}}}}; }}"),
                ("src/services/weather.ts", "unrelated", f"// {repo} independent weather integration\nexport function getForecast(city) {{ return weatherClient.lookup(city); }}"),
                ("docs/changelog.md", "documentation", f"{repo}: landing-page icon spacing and release-note formatting."),
            ]
        elif stack == "django":
            units = [
                (f"{plural}/views.py", "controller", f"class {entity}CreateView(CreateAPIView):\n    serializer_class = {entity}Serializer\n    permission_classes = [IsOperator]\n    def perform_create(self, serializer):\n        serializer.save(owner=self.request.user)"),
                (f"{plural}/urls.py", "route", f"urlpatterns = [path('{plural}/', {entity}CreateView.as_view()), path('{plural}/list/', {entity}ListView.as_view())]"),
                (f"{plural}/models.py", "model", f"class {entity}(models.Model):\n    class Status(models.TextChoices):\n        PENDING = 'pending'\n        ACTIVE = 'active'\n    owner = models.ForeignKey(settings.AUTH_USER_MODEL, on_delete=models.CASCADE, related_name='{plural}')\n    status = models.CharField(max_length=16, choices=Status.choices)"),
                (f"{plural}/permissions.py", "auth", f"class IsOperator(BasePermission):\n    # {entity} access contract\n    def has_permission(self, request, view):\n        return request.user.is_authenticated and request.user.role == 'operator'"),
                (f"{plural}/serializers.py", "response", f"class {entity}Serializer(serializers.ModelSerializer):\n    class Meta:\n        model = {entity}\n        fields = ['id', 'status']"),
                (f"{plural}/pagination.py", "service", f"class {entity}Pagination(PageNumberPagination):\n    page_size = 20\n    page_size_query_param = 'limit'\n    max_page_size = 100"),
                ("weather/client.py", "unrelated", f"# {repo} weather service\ndef forecast(city):\n    return provider.lookup(city)"),
                ("docs/changelog.md", "documentation", f"{repo}: documentation navigation and release formatting."),
            ]
        elif stack == "fastapi":
            units = [
                (f"app/routers/{plural}.py", "controller", f"@router.post('/{plural}', response_model={entity}Out)\ndef create_{entity.lower()}(data: {entity}In, user=Depends(require_operator), db=Depends(get_db)):\n    return create_record(db, data, owner_id=user.id)"),
                ("app/main.py", "route", f"app = FastAPI()\napp.include_router({plural}.router, prefix='/api')"),
                (f"app/models/{plural}.py", "model", f"class {entity}(Base):\n    __tablename__ = '{plural}'\n    id = mapped_column(Integer, primary_key=True)\n    owner_id = mapped_column(ForeignKey('users.id'))\n    status = mapped_column(Enum('pending', 'active', name='{entity.lower()}_status'))"),
                (f"app/dependencies/{plural}_auth.py", "auth", f"def require_operator(user=Depends(current_user)):\n    # {entity} role policy\n    if user.role != 'operator':\n        raise HTTPException(status_code=403, detail='Forbidden')\n    return user"),
                (f"app/schemas/{plural}.py", "response", f"class {entity}In(BaseModel):\n    status: Literal['pending', 'active']\nclass {entity}Out({entity}In):\n    id: int\n    model_config = ConfigDict(from_attributes=True)"),
                (f"app/services/{plural}_list.py", "service", f"def list_{plural}(db, owner_id, page=1, limit=20):\n    stmt = select({entity}).where({entity}.owner_id == owner_id).offset((page-1)*limit).limit(limit)\n    return db.scalars(stmt).all()"),
                ("app/weather.py", "unrelated", f"# {repo} forecast client\ndef forecast(city):\n    return weather.lookup(city)"),
                ("docs/changelog.md", "documentation", f"{repo}: build badge and logo documentation."),
            ]
        else:
            units = [
                (f"src/components/{entity}Form.tsx", "controller", f"export function {entity}Form() {{ const create=useCreate{entity}(); return <form onSubmit={{create.submit}}><StatusSelect values={{['pending','active']}} /><button>Save</button></form>; }}"),
                ("src/routes.tsx", "route", f"<Route path='/manage/{plural}/new' element={{<RequireRole role='operator'><{entity}Form /></RequireRole>}} />"),
                (f"src/types/{entity}.ts", "model", f"export type {entity} = {{id: number; ownerId: number; status: 'pending' | 'active'}};"),
                (f"src/auth/{entity}Access.tsx", "auth", f"// {entity} UI access; the API must enforce authorization too\nexport function RequireRole({{role,children}}) {{ const user=useSession(); return user?.role === role ? children : <Forbidden />; }}"),
                (f"src/api/{entity}Api.ts", "response", f"export async function create{entity}(data: Omit<{entity},'id'>) {{ const response=await client.post('/{plural}',data); return response.data.data; }}"),
                (f"src/hooks/use{entity}List.ts", "service", f"export function use{entity}List(page:number,limit:number) {{ return useQuery({{queryKey:['{plural}',page,limit],queryFn:()=>client.get('/{plural}',{{params:{{page,limit}}}})}}); }}"),
                ("src/weather.tsx", "unrelated", f"export function Weather{number}() {{ return <span>Sunny</span>; }}"),
                ("docs/changelog.md", "documentation", f"{repo}: marketing colors and typography release notes."),
            ]
        for task_index in (0, 1):
            task = f"Create a protected {entity.lower()} flow" if task_index == 0 else f"Add owner-scoped pagination for {plural}"
            labels = [2, 2, 2, 2, 2, 1, 0, 0] if task_index == 0 else [1, 1, 2, 2, 1, 2, 0, 0]
            for index, (path, role, content) in enumerate(units):
                rows.append({"repo_id": repo, "task_id": f"{repo}-task-{task_index}", "task": task,
                    "endpoints": f"POST /{plural}" if task_index == 0 else f"GET /{plural}?page=1&limit=20",
                    "access": "Authenticated operator" if task_index == 0 else "Authenticated owner; only their records",
                    "rules": "Validate status against the existing type; use the current access checks" if task_index == 0 else "Preserve owner filtering; paginate without exposing another user's records",
                    "response_shape": "Use the existing project response conventions", "candidate_id": f"unit-{index}",
                    "path": path, "role": role, "content": content, "label": labels[index]})
    return rows


def validate_rows(rows):
    if not isinstance(rows, list) or not rows:
        raise ValueError("The dataset must contain JSONL candidate records.")
    if len(rows) > 100_000:
        raise ValueError("Start with at most 100,000 labeled candidates in one Colab run.")
    clean, seen, queries, grouped = [], set(), {}, collections.defaultdict(list)
    leaks = re.compile(r"(?:gh[pousr]_[A-Za-z0-9_]{20,}|github_pat_[A-Za-z0-9_]+|-----BEGIN [^-]*PRIVATE KEY-----|\$2[aby]\$\d{2}\$[./A-Za-z0-9]{53}|(?:postgres(?:ql)?|mysql|mongodb(?:\+srv)?)://[^\s/@]+:[^\s/@]+@|\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b)")
    for index, item in enumerate(rows, 1):
        row = dict(item)
        for key in ("repo_id", "task_id", "task", "candidate_id", "path", "content"):
            if not isinstance(row.get(key), str) or not row[key].strip():
                raise ValueError(f"Row {index}: {key} must be a nonempty string.")
        for key in FIELDS[1:] + ("role",):
            if key in row and not isinstance(row[key], str):
                raise ValueError(f"Row {index}: {key} must be a string.")
        if len(row["content"]) > 32_000 or len(query_text(row)) > 8000:
            raise ValueError(f"Row {index}: split large files into complete functions, types, or model blocks.")
        label = row.get("label")
        if isinstance(label, str):
            label = LABELS.get(label.lower())
        if type(label) is not int or label not in (0, 1, 2):
            raise ValueError(f"Row {index}: label must be 0/1/2 or irrelevant/useful/essential.")
        row["label"] = label
        text = json.dumps(row, ensure_ascii=False)
        if leaks.search(text) or re.search(r'"(?:sample_rows|sample_documents)"\s*:', row["content"]):
            raise ValueError(f"Row {index}: possible private data or credentials. Sanitize locally before uploading; this scan is not comprehensive.")
        key = (row["repo_id"], row["task_id"])
        unique = (*key, row["candidate_id"])
        if unique in seen:
            raise ValueError(f"Row {index}: duplicate candidate identifier within a task.")
        seen.add(unique)
        if key in queries and queries[key] != query_text(row):
            raise ValueError(f"Row {index}: all candidates of a task must have identical requirements.")
        queries[key] = query_text(row)
        grouped[key].append(row)
        clean.append(row)
    if len({row["repo_id"] for row in clean}) < 3:
        raise ValueError("At least three independent repositories are required for train/validation/test separation.")
    for key, group in grouped.items():
        if len(group) < 4 or 2 not in {r["label"] for r in group} or 0 not in {r["label"] for r in group}:
            raise ValueError(f"Task {key}: label at least four candidates, including essential and irrelevant examples.")
    return clean


def dataset_hash(rows):
    normalized = sorted(rows, key=lambda r: (r["repo_id"], r["task_id"], r["candidate_id"]))
    return hashlib.sha256(json.dumps(normalized, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def split_repositories(rows, seed=42):
    repositories = sorted({r["repo_id"] for r in rows})
    random.Random(seed).shuffle(repositories)
    heldout = max(1, math.ceil(len(repositories) * .20))
    groups = {"train": set(repositories[:-2 * heldout]), "validation": set(repositories[-2 * heldout:-heldout]), "test": set(repositories[-heldout:])}
    return {name: [r for r in rows if r["repo_id"] in ids] for name, ids in groups.items()}


def training_pairs(rows, seed=42, maximum_per_task=32):
    grouped = collections.defaultdict(list)
    for row in rows:
        grouped[(row["repo_id"], row["task_id"])].append(row)
    rng, pairs = random.Random(seed), []
    for group in grouped.values():
        local = [(query_text(a), passage_text(a), passage_text(b), float(a["label"] - b["label"]))
                 for a in group for b in group if a["label"] > b["label"]]
        rng.shuffle(local)
        pairs.extend(local[:maximum_per_task])
    return pairs


def unit_cost(row):
    return math.ceil(len(json.dumps({"path": row["path"], "content": row["content"]}, ensure_ascii=False).encode()) / 3)


def select_under_budget(rows, scores, budget=2000):
    chosen, used = [], 0
    for index in sorted(range(len(rows)), key=lambda i: (-float(scores[i]), rows[i]["candidate_id"])):
        cost = unit_cost(rows[index])
        if used + cost <= budget:
            chosen.append(index)
            used += cost
    return chosen, used


def ranking_metrics(rows, scores, budget=2000, k=5):
    order = sorted(range(len(rows)), key=lambda i: (-float(scores[i]), rows[i]["candidate_id"]))
    gains = [2 ** rows[i]["label"] - 1 for i in order[:k]]
    ideal = sorted([2 ** r["label"] - 1 for r in rows], reverse=True)[:k]
    dcg = lambda items: sum(gain / math.log2(index + 2) for index, gain in enumerate(items))
    chosen, used = select_under_budget(rows, scores, budget)
    essential = sum(r["label"] == 2 for r in rows)
    return {"ndcg_at_5": dcg(gains) / dcg(ideal) if dcg(ideal) else 0.,
        "essential_recall_at_budget": sum(rows[i]["label"] == 2 for i in chosen) / essential if essential else 0.,
        "irrelevant_token_fraction": sum(unit_cost(rows[i]) for i in chosen if rows[i]["label"] == 0) / used if used else 0.,
        "estimated_selected_tokens": float(used),
        "mrr": next((1. / (rank + 1) for rank, i in enumerate(order) if rows[i]["label"] == 2), 0.)}


def keyword_scores(query, rows):
    """Independent keyword baseline, not the complete current Contexa selector."""
    words = lambda text: re.findall(r"[a-z][a-z0-9_]+", re.sub(r"([a-z])([A-Z])", r"\1 \2", text).lower())
    query_words = set(words(query))
    docs = [collections.Counter(words(passage_text(r))) for r in rows]
    average = sum(sum(d.values()) for d in docs) / max(1, len(docs))
    scores = []
    for doc in docs:
        score = 0.
        for word in query_words:
            frequency = sum(word in d for d in docs)
            tf = doc[word]
            score += math.log(1 + (len(docs) - frequency + .5) / (frequency + .5)) * tf * 2.2 / (tf + 1.2 * (.25 + .75 * sum(doc.values()) / max(1, average)))
        scores.append(score)
    return scores


def evaluate(rows, score_fn, budget=2000):
    groups = collections.defaultdict(list)
    for row in rows:
        groups[(row["repo_id"], row["task_id"])].append(row)
    metrics = [ranking_metrics(group, score_fn(query_text(group[0]), group), budget) for group in groups.values()]
    return {key: sum(m[key] for m in metrics) / len(metrics) for key in metrics[0]}


## 4. Load the bundled public pilot, demo, or reviewed records

The default public pilot is embedded below with attribution notices; no upload is needed. Its 0/2 labels are uncertain unchanged/observed changed units, rather than verified irrelevant/essential context. The following schema also supports a separate reviewed dataset.

**JSONL = one candidate record per line.** Every task needs at least four candidates including an essential and an irrelevant candidate. Include difficult negatives such as a similarly named controller from another feature. Provide the same requirements on every candidate belonging to one task.

Required fields: `repo_id`, `task_id`, `task`, `candidate_id`, `path`, `content`, `label`.
Optional: `endpoints`, `access`, `rules`, `response_shape`, `role`.
Labels: **2 essential**, **1 useful**, **0 irrelevant** (names also accepted). Candidate `content` should be a complete function, class/type, ORM model/enum block, or short file—not a directory listing or database records.

```json
{"repo_id":"project-a","task_id":"interview-create","task":"Create an interview endpoint","endpoints":"POST /interviews","access":"Recruiters","rules":"Reject overlaps","response_shape":"Use sendSuccess","candidate_id":"interview-model","path":"prisma/schema.prisma","role":"model","content":"model Interview {\n id Int @id\n}","label":2}
```

At least **3 repositories** are required for a disjoint split; use many more for meaningful generalization. Before real training, have a developer review the complete candidate set for each task, including the shared auth/response contracts and relevant enums. Synthetic labels and model-generated labels alone do not establish correctness.


In [ ]:
BUNDLED_PUBLIC_DATA = 'eJztfXtX20jS9//POe936GFnxvaskS3frYRkCZAMZwjkATKz+4QcIUttrEGWPGqZyzJ897eqW3eJYBzDkN2eIWCr7636VVdXV1XfrM2MYLKmkTXmm42xY7DzhunYyux6rU7WTM8NqBtgsglJjBzRmeEbAbU+QKnj6xmtQmbzXMGvNe3EJfDfyRr+v4XPiTcLbM8lAeQkwcQIiGGadBYwYhDHZgHxxuTCcOaUERbVTEbXkJWKug6OKoxgD6N0zyfV01Pt9LROTk9fnJ4SqP0327W8S1ZTyI5hTkSFxGaiBvhmW7xeA1vV+Di006Tbp7x3SrrvJ674ZtExgSm4oH5QFU/wP0adcV20opFA2XSv6wT7NtVIVCt8oQH1yZ9k33NpnZjBVZS4hVN6FYRJotYaWX8latKSZuyAThnZ4M0pbObYgU7diwvD13nTVf67luT/GzENtxLAb8chrrdu+GeEzWfUr9aI7TLbomLKTW868+mEwiOYu7kb2A75cB1M4EtbUVupcWJhPRw/dkRUpoRPkow+Dea+Sz5lClSx/+HE8AmokTG8PHwK3RGj+3ziIpU5xog6QGMtpDjDtfj70m2rjCq1Qa+vDfpDLMe8uW9SnQWGjyQKKclD6mJxyAiPfDrzwupmMDk0YKJKrCOAv2FaT+0Nokf43fK9GQ565sHwse/JLDUxn+85lHeRtxgPRA+/Q8olNc51oF5zonswJ44x0y/UOCPT4Zk9tim2PjYcRpPuw0ua2hx3Vr89NlTTMkfWYKR2el1T7arWaDBsdUZGFx5Zltqh7W5qQuY+zubaJAhmTGs0zuxgMh/Ba5s2MsNvjBxv1Fi0/kYZf+Czt2hzs7njNKJJBiwABVJ9bItJDLPqOs+r64t2SwmugrXb//c/N4vyMYnrZ4rrfvsuXPfbEtcS10k/J9QBgmR3Y9s3bOiC4V5XBaanlDHjDFDNAp9jEiGagqRY9Q+xFLHHBAoS6vuez8gl9SkgC8ZgggihROJBjOWxgLHInaoxSrxmChADIlK33bFHXm2QKlCzqtZymTneeAfeGIzuXKGUBKXe+d58Vg37X083VkPO4Hp/GBp5O2ip2dqACOmdDaQq+dR8EFSTidd6/Z7WG7RK4AopebhCRgnXlcM1C4PnDVkGFAzk/AXMnlFYB69mNsj4iJbAntIQvcZsBjSO1SAAeD0abET4h/f2le1ySCPBYqFwAS6ge5OI+QJmAORjid2IWOFgX+CSpG1eFac02IZELSbVwfPTU2wCNh8A8VQe2HPAUuyew14j8HjCyId9CbQZZlAIOY52NuG4jbkTEHs6c+gUJkS0H/XK9S7J33k90O+p4UKGYncceyzGDXM6ts/mPrSOOy4oBRMH75bXqeSnI8fDRPfiZnK8IxQEoilWoGNV/PBvmAVlHpg16CY0lpTXwxr1qHcFoUIISYuznhQBaa1WW2u11RLeAyl53gMZJe9ZOe/J4fl5M58v6DP2vc3ZLF5tQ2XGR1xsd3CBzKo0uIBgCQkhDS4U0F0vICPkGnPXQhbheIaVEhjShN5cRBxu97X2oITC24U9bnsg6fu/VhQuJe1/BAp/F0CCJy6urY5nIqkByYbb3alnzR2qu7Cd5UIxX2STr/H+VgjS9lgH8tY5bePOeA82eb7hfDr25/QzbCDx74lYhfkyrRFFUZag+VZrCHy9TKaElCJflzKlpPu/hO7f4gtFwudkntB9JHouS/7tLvwrY/qQUiR/yfYl+WfIP9pFhQqHalE5eZc+M6v1zGtIQN5B6UYcqwCxQL2KT5nt2CjpQ43Mds+0vIQfnaGIUYd9AszAXgeozrFHCjRtAB6MSENSPVnjmU/WQu3nJfXP/03nZ4uWjvLzCkQVYqDUnHgpRe/4ZC0k+JuZYwQAgCnMJj6I5652ewJVrGXKCIzfZAZUku23sBfkJt//25O1epLV9BzP38DJ5J/ClHDo+Jhe2UG1tgwj6fW11qBZxkiKRySQUTISyUjyjAR28AFfOO/lIgd86xSzkGQZ5Xwk+Zpwk+iANq+oyagGwtyoNN0QcHDZ3Ke6N/qdmkH1yPTtWbALybUkp4KruWAROj803hBtiRxh9eGTB6Oq02nCv14JqiAljyrIKFElUVWKKouO5mcPxtXI8xyOKPyQhdTfyK7QQ0LrZ4ArPC2c+d6FbVGrTuyAXNqOEysbA49wAVYh2x7mnDMaVYM60TqB6SC8j6jQgA6jQQZ1L/B0jytNTYNF5hLivYcARdljFokUITFW+QMFJex4TUaFoyhnMy5Z4FAIwC9+7JLUah1LKUc8Vdneebv5ce+4fm8O/f3mh2hV/TKX+Rs5ojhLwDFwoBeGjyerATUsNFBJWI2CWwvopZgHG6fChUmKKkHbFoYaImr4zjWx5j4IRsQAiJmB51+T8dw1U8pYjwFPu7B9z/0EwsXe5tEv+vbOm4/vTtZwb3Gypp6sJdwST3TgWTMSNr6WmfUGWqf0+AZSCsxMHt9IZlbCzDCvbjPdcIHugcKr+CDcV3vAj/zkwBWZVkaXemycU3J6yrOdnnL0+3QK75czMoe6ZwBGAN/pKVYKOca+NwXIKXiC4pLfPW5rIGoEjhZlU5AVAjCBs2A9nm+f2a7hCJzUk9psfuYTdRwaEjXFHcqc9IZYA7xiaQXb5kMNR/kJesu/14j2WXFgyPasCpkZndVqZGND5FoCpL2hCv+6JSCFlMIZ67ArQSpBmgcpm3iXIHX4MP/6yHBd6lf52i4ECqH/SknMBfk9JWYISBxBhQSkFd8gnBrns8i8QkBubPsMhAw8kRRHo9g04s2fu+FqbJ+5Hl8aMYNP+VGJnzPADAUF4C5QzoXMOjIAPcpdLa7oKfEivw1IyRmpYindAGzbyU8gBPgX2C2x0YdKSOUmV9VtJaVcgJaEnLRg/ds889SzQJy7qYDwldTAl/eKNx5XbnkDD+YV/V5P6/f7JbwCUvK8AjJKXiF5RZ5XTA1Y2PKox9wiYQmqVNVWC3+1S+gS0/KEiVklZUrKzBsI/UPw0pnBmG6K3XJioW9R0+OeAaV76kad/GT4Zyw26f3pp/PL1INyG12h8jbnvo86brRAylrE4OJwj1qKbxa5Cq2WLYulcK8Ji1m4SUbbGVxowqFVa/ESkxI/sZjtXnjntDoOx5QMZhlwtocdrdMsUxNDSmEP2JRq4v82YFq/G+6Z15gHtsMapmFO6B0bwcDTQQ50aJVFYhm3KxfW7dWTtY0TKKXWYqkJN0wB3xq1ioqZ4FMTdlHeJQp5dfimfs7swrLJwvTgPjOy0oFonU5f63TLzMggpUD/3ZwZWVin+JOj/1ZL7XTSAHhrX1GLkPV1Ij5dTuyAsplhUjKBvjoodV7ivjY8TYOl+GodRGpi2T6wFfuCMuXpgAICNh2rzYExNvpmz+w2h92hOW6Zw47RsTp0PBqrY4MO1YWAkpknAZRF62/cSYD3oCXbJkdL/ErycBGZACbh30U7tzxcUE8K71eH91uFJWDmQXeyhmbi82FI8LBJisiBK0HErkmUI1vYzjqudkAbZEJxc8Q93VxiQ6tnNDpYBK7L9aw2149cGlxLHJurhd+TUuXbMVwUo6aVicF00SBAPNMP2MKUbM/42mcCY7BsM+13k7AP6ghnEuqgDlj4xogWQt+YpG3+lH0qNPy5ljlHDfzrIoeBIVZNE8qG05qUotwSkFR/xda4BSCwmOtZ9PEXep0xC8T/UCa5f/Et50BqswfScZnPC6QUZeOh5ECSA62CA+EpVAQlnQbGWYEP5bHOAp8aU36cwTW14eOw3gzECgjdOTbOxJnGH3MvCBvMisRTq1vNV1rHkykL2AGjIIfbwfUGP7aqQc1Xln1GGcjJqUOgWkZKiCpbGpm4b22VqVPKdq2tvkSmROYqkKnP4J16QKHcCWZsQEcsQMYfcyD3CJvx2r9BfobxHYZfq0ChwZxtdNRWiAXHO4thngLcydqHVCPkLW9EIz+wjHlS2Cg/8qgX8b0RfSgW2Qj/Zi2aVgXMdhP+lRlJQkoBmHkjSQlMCcwlgWmP9bk79Sz+InR43zCBKPtRVoV9Nkj0YVKdlOQsE++PKZ6TgCy/O17/GOdfP8L83Jnb8G0G+ARhHjpgu+h54pLDt1tkqKpNckS5LYGoS20rqtIpkdlD1GV6yJfw7JOXG2W9XhajrfZQa4kXVzBkLtrxdzoSoxKjq8KoC3tc2F3jawrRCaTHfUtB6qwT/M3uQyPuk9ffYx2L41BAsHX3tjnVjZS4HJlu+TS024IdOMrLdXzoZlMIH5ZS2M4KHRjfvjrQGB8i6tZA9P4J5O5Ma8cwQnplM5g2gIg3jtojU2q4TNg1RY3CKIxIBZ6uA7AKq3jsrRo5yooBkkipjSo/bikVVhZ2H401Kj9ViqPgwn00jIz3uuh16k2wiTd3LLROQ1DyZhzPO+c9iZoxxhing9tizAD36dq8sXDxx8LwKi304oNdRqpLqXcFQlbqmyKMO07WfmvEVuq8x2LOSbWYSSg0sCagGkF+RZVnqj2klSjn0gy409davbJDt1ZRswkZJQOWDHhVDHhREelhAtL7rxeP2neLR4i5rECEHryZB6+KfV4anb2e1uqXuq8XTTUgo0SnROcq0CneTqz3CxV0sd6tTnhZHnnDmwcb3ASsBJablkV+Pj7+wLMjzYQVRTEvzoBq3IxmQiM7GFyD4qmEJSrJaO2V6LSbh+8LzzBAZPBc55oYliV87rlVJxeEHB+yXKMWMy6Z6TqWxfMDruRg3GKUbG1u/byjv9/d3t7b+W3zcEc/2tk62N8+wloCHAWUQX2jqG50HZnLlwt0hebypl+Z9I2oEabc1Y07an5Jml+otYnSzBYPDjYxLpJTIpeeGYjdBY5twheTP7Ap6nCjfKjGRcrXkc3xICAKD9WCIUAy/QulHEF0IsUUrztFceEx2Ea+5JKMtd/RWsNSv7d+waABMkrGKhnrKhgrsCjdpWhhK+g4z1nv4qIpxmkkWl34Fm5KopCpQERAOeGuhzeEex/ellXCnx7I59fVL0C1yBTqwE1Epg3cePKvaEMffZ3OGR4uRWFXw6cz376IvsVq4SVRPmxr7VKzJUjJo7ydN1uSKJcoXw7luG5eGP51tHgmgAhNFP6YU/+6DOtbE2qeI64ZpVHIsBjuEx6WWAhNoeiD3nIiliYjv0KL4fOHmmRg0TJLjIyqBXVBaF0f8aINcjOJzKu46mKyuCmGaPBz7TazuUrPTlRzqCKpcU1HrgvL8oV2uwX/yowp2u3CkW27LY0pJF9YkdJDHbjhsnlOr3U2H4/tq+jQNlptISXLGnYBuhT9RQE0dRQhOFsI1a08co3Do21wTxoMtBftr3h9BOpTcpwg3mR8PNrRd9XBfkaHKvxyTM6KIPMeb+C9bVkOvTR83pThck81Mk2e8t1Xupq9zf13Hzff7ehbB9s72KV4nFxhPcYgxSNDsDseg5CTO3FgLudoapWqClBjcu4izJvRH89hXraKeEyZdvNbIpgM8nd0mVXwEJv8gCZ2RhD4yUs4WctWAO8RNa5Rt7jFddk0Hv9fYf9V1pgeWYpHURF5TKFq9ug7Lr40h+sAhys1WIWUAoeTBquSw32Zw8ESHuhj9J2/9PzzsNdTatkG3ohQEqsV2Z3n42o+uuaWKtTCc6QqL6NjId1J7FTS7OkIRRqQdE5FcaQaamVrOa2T0RzPyc4pI3NXZCG8anF/BMoMiMti7Un7TD+jGCeoqr/HZ/yGimkgJJkpP1jJ9TXr4Xtn56q5Jmr3G53fP7tap6t1yzYynUK4r25uGwN98iwaEsw6NrUeN5XDttps9nppbB9yn2cavYvsSxAmxm1FHSgqMmBYyjCK7TmNwtIKV+qnw3ZfHXbHfdoxu2q31W92O71Wmxrd7tjsdIZqf9AzTHOojgcLYftL0yagvmhzjcXAcw/uv9gfzgbit5dnA6JoxAZyZXV90XGsiCukUMXPX/HIZ8LqxJ+UnrYfhhGOT09RIZAEUk6h3XZT7sSGfzbH+MhoqBLfDVPMLDS+cW6Fh1tO5YOdlh/zEVEJPz5ipm+P4ptnhHabzagZq5d/RW1GCS+Chk1nbqHqV2SspMLCNn5nnvsClTnQl41O5Y4c0XP0/Gr8FH37Kf6YnjeYU2BuKd4GDyIWlk/yk6TQCGjCFP5ywqT7hJAFGBja+ZTwrxIjH8m+JPt6vuxrsYW/jJO944obg1984hsjhxvVhLLL6WkCR7ywKqTA5Oor2Gqw2D0HeiBYUNK4Qj7En/GUycIgR1N+1g35MBADcimgQRPtXlLskMUMqa0AZ4FnDTYf4d8XIsjUBsAg5C+tXI7wsRo9jvhQU4m4kvieHR12b2KwCZ8C6BzqruLoK4JbohaYsjDGgmDXwfrIwJmCTeZ8Vn4+D0ztE2REZ7+SP6FvIBfv4heVFfNYVgH2qU3WU6lKMtufFdjtpt53lnt+Gs3NcyrcV8OPNmcquHaJB59XwlNbfa1TGgO56JPYklxVctXnyVUROVw7ptuuHeh6fBNPLCdiaKgUNPmtNMgxAPAVHhmlms0cHbPXRLCUbGLKdi/j8RfXPIZ547nDS3Q4F0SJifPqSLUdh69j1bg/aTM/4R6YeAfmWvob2eYTL+JHs1RGYowwttX7zX/qP+9sbu8c6ns7+++Of34Bk3TmQ9v8EDBfW4ql85PAULRGnT+3vPANrq+DNJewuT82TBFwrttsKg+Zg0qlTm5uc+8Cg37EuWdhEVgjBIvdyFWJlQXcoaRaaVRWI112O1qvjBN2C4f5PckJJSd89pyQ/hHzQa5oLzmasxkGnsS4dFWepZ7a0dVEjMyET4aR5fi3lQCuB4ArDTdXBFxXAk4C7rkDDjckIeRKsIapmWV+FQjqa70y58Ri3PXeQCJIIuh5I0hoC+9YscI7sUIPFn8emraAlB6vWIQZgc2AHljKMjhOLdymiVtrPF3ltruxfFq8VRPzfAdS6x8VviKKRVBkxiDUVUivh1uF7zbSVeVSy27IzFjIJOnRRZ+x/Ivt/5Ruf7GUTCXlt+zd2XYskBebWDQpW8+C7WfczFbAI/tNrTTydL/gv52POy15pOSRz4lH/mPmezPqB9cCLNzJItEc58WOPLs7jNTBSRni0AvqCGpsrrdj38qUIiC0FEzrfrlnxH03fGYYyAbnEuX4b6b0HU6B732ppJoriaaJaUUDOlXZLKimnvEgcJ+Ak38ur7JVYEPtlfCgQUcrDaw9KOx08mG1JQ+SPOg58SCx02GBX77R4TdJ/MAaoa1atajaS6G7VpDF+PUjOYFMsQM6ZdW8+ISNcbu4F+QHthG1F9VRdHqGP6s5g+5rarPMKG5Y2HNBPglmCeYnATMGeYVRONQMWMMatRrCAP53lsMwfGQBwXEDVNFe1fZptYLfK7UX99mY3dmI1tbK3P+L/jFZPDC0l3Xsf9NG/CkPg0G3nYEB2jpDyzWN0/j2qEUC9NvG0FVzdPmc2BZAAo+EcdC2O6frnrtO8VDk6ah/aLap2TX7/a7VaQ96fXXQGfdag+FoNBiqvc6gORzQ4aDdW4j6SyZJEP2irTS+SBv30HpZ64LEozeTJ/E4HywTycdFO/u1lD0ChE2mhn+OkTQmNl6LEvZB8fgNV0xJsvz5p8hTSPkaJAAUSrFQBINEg0TDY6EBb1XjSW/ome1+DT134KeMngtbl3ZH0rOk55XQ8/oZdSm/ZOEuPq+nxRfHsww2yQkwd11KcE9rUpT59oi9QC7/sWT/ETcsadJXlAb88H3Miui/o5Wx++INHZL+Jf0/Of1vGwE/zCvBAN7Ovs538SsCQlcrU9kWfaUkECQQnhwImyMW+IYZ/C9mfxflzqHCCDPlK10RQHpa2fXkhZhyPQkQCZAnB4gPBO1N31wH2bXC9K9ngVepKan0lYChr5XehpkHQ1+CQYLhycFwMCvKS2hKwCMRr2g1GGhlloiF290HEgASAE8OgGDie5fvaTDxrI9uFMB3g4znLo/bW53ypH1jSmvk5oT79/AixKWXhLvUVE+FczfmIydr398kRW5P1qI7mcOqvyMfHGowHtAG7WUCj1zDSybsD4eE3VdOay+woduVgE9VNbX81ts8/lSpuJIAfHoA8myAuNOtzb09cvSvow+HB1vK5vb73X19+/Dgg3609fPO+83q9zfk0jdmR7Z75tD/xQu8qsyc0KmhBL49rdZq5LZO9j/u7dXJawL/aqerUfr2tHbZZqZd2M205XZG4ufp8YMrxwa5CXG0mkWj3YefMqIv7Fractsiif6vWzQqW4c7m8c75Hjzzd4OefkDXpKCgR9+eEWq+A2D79mjOW71f3hVq9RPYgftGSwahn/9C71GLcCnz6mksedT+8wNk25uU0lY31Hg50sgnUD/jul05mC4ZOgYhqzDHoRJ0HwdHv79xE8bK1aOsbcoKUKRdO/rZMtz5lM3lWTyB5D2orIaJXZX65XF/OoW9Ni9psS4xPjSGJ8Z5rlxRlnDBFzxUUXa5/VweKkTGiW4w3prPka7LcjijYXJKtlAFwAR0aUCIt+b+XhMfQXtXas8A+BtZLuA8UqNaKLMvdYuD+2rBtMw4L/KtljwuLDJEs8ejKZmDk0aocw0ZlSMK7wwy4yuo7LhffoBv4kGNphIN+j6ZrB1mz0dnPrGsN+zuj21pXZNc0RVy+q0mrTbMds9o9ceqf1hh1K183VwWrSVxjJk+DXYai6OrUXHsBS2FlxRQzqCJdNxNuNVU5kaMzSdJxuvIu1H7MpT5RaSkdKCzWczzw+YMpo759vizhLy448ECBFysE9QyWcOWfTQrGUrI6TRgIWBOhYPVPKzwSY0LPH6Nals77zd/Lh3XMkWCc3Xs8VEoe82iAuvAZhCWaqW1PgiXeUtfkk/iC5Fw2EKtFXL6oMuYmv1kjFNPYs6+OH4YPtAI1PjnBKcn12OT35CRwyTx3IRWTHoAjUsjMmVn49C5cDcrA/oCJE0EL4EniTia+WKcW9LUfV7YzajVvKq+WBeK9wbIz/TINWYFMUYppHIJDb9NFPgtpaa1tv7bceXomHYlA+09qBMgGn3ClwXMkqm+0yZ7uqlmm+M83IYTTwHb1wAieZk7fsPe5tbOz8f7G3vHH5/skYEz6w8EpDUVrvFf5VJMK3ijQXhMwkmCabVg8mD0ThibDq8eeq7hpMSy9bxVfq2VbpPcAzGyOHO5h6hV/DQYuSNwSi3CVP44+j4xjv6w6mmRJBwna+82d3fPPyXvn3w8c3eTiQbpKQCWGaj2bu8vFREX/kMzhzWCLyZbTbw/tf57LUZXG1YI9QFsOBH29p493F3e735ZrO10+x11wfNZm+90+ptrW+2t96sq297b5pbg7etQXsr6uIbWMFRF8AFALGf0fCy2FS3UQRLdkL78+kItj8fDo52j3d/3dF399/u7u8e/ysjakUjtd1xIvtk5J7SSvd33m3eX+n63bWmXQ2L86qf0QDHC1JjVUyqNdLInj0qvqObUIKJsinbb/Tjf33Y0TPvjtymG7ldmG8+hPq0Vm941yVvvWLc4fwlb5JnPjHPfChn+XZ45nJqFXQDeu+h9M75yr0qxQdrRtoon5eJ5yXSucTGM5Un/iM1Il+CRWT3QWwm0JFeezUy8oBQDDdal8JVSeR8rWTL1FAzwGnhRWEZWgnCOl2tU3ZKV7y1o7PUKZ1EmETY6hHGo/ds4c7Xn5tALy+PX8EO9zgW2qPKuIEViOkbr4CixrA33g0DkpLXyUeNHK9+5eqqWunlVt2CyVT+aiuJK4mrvxRXm1yZjBtH3PcejF8muIocpABtYvHKYTDKF/mPRPlfCvQlNb8CACbfAIL8JuBHgGFb65Y5HHYLHrfdpVwOJQwlDB8JhhF24nVqowgrEChfPQJoevBTBpqCuWJ3KXNFCRoJmkcGzRZX5G4Q/vdlKW4eAzh9+CkDTsHksSs3UxI4zxY4B35q0SmsQ39mMz8CjIbwUwajgkq8O5QwkjB6hjDCK4W4VWEeLQm0Vo+bngo/JbjpFXQOPalzkLh5TrjZ9/ypgR2xvoSgR4ROG37KoFPQE/SW8nCU0JHQWT10+Bn02ACZ7E1kx3kgLCujw6XYvjM6gpq756536da0EF2PdKzU62q9sp1Qr+gYIndCEk+rwpNjjxoT6syozxoz3zMpY1ue61J+HBtCo2hCGZ/X3lWkOvftemS0HBsUCctLeIqKhjCRvI4/aeQmMiEKY8ZQw/rg0zH1qdhZYVEl+zQyVXr9pUSNnNHgo28fZlKxk8JqOt2o6XgutQ5EJ/mXKtZcy/WMec5FnK2a6+mPP+b7/h26zYTeb5V70sVjn1qVWjK6sEVsEGZ7DHQRsFyz9VTfa8nQk4epscZR+JNxFBlbmdnWggSj9TS19Ca0PDdTc2Yom/PAmxpBYJuNqeeeeR4rcLNed9jKcbOqGfcBGLXluZWAWDbjRyUG1LjrWvSqgZ+2YM4Cyi84MCJ3RJKlCxJdwvh0rM7o9Ptj0+iaHYofVXXUH6tGu9uxVKPf7vU7nRYdq/3F4vOXTKFgdYu20ngAV7iHw5X1RXC46C3mOVxSQtejIrq+aNdXxeGekAGtCnEDTS2ziyw6wzUl4iTinini+GKVCY7V4I8WCIq1KE6WCygtUSJR8mxQ8j5sX1wunw0lx933GzYS1gpBs1wUagkaCZpnA5ov7ztXBZTl4o5KoEigPBug3KkBWRVGVHR9LUGJWuL2KnEicfJMcfKNKgFXhuK2ppbdyKAWtlJqV6JYovjZoTg+xSiF1QxohR9hJOcXjUZIdgzG4QbONYn8V8mcUX8dmr6AL3hqBqSIJ/GhiCmKo5c3owHjhBodlIgGeMiegB1R6J4F1IrEyntQS3lj8wjIWTIWoZBrpV7z0Bz2ikwMEQuZXs1gZDZ2G9pC1zoOm/V8Z+txDZeUAHNIRmuFAYaCCZ1ixCJ+tS2SuygBQKAF3/GwTDRmduy9xRKZM5Wiw/jX8KVWU2uXGfW0mgUVjyr5kuRLz5cv3cEsMqep8Htsn+1aV9HphHjwWolJL3NweUMSQqwn5Eluw+IvMseD331XTer/889U2egbL11bzCZiYQVt+444uEUdrdxGSwA/PwAvjMyVIWbpS2ElYiRing1i7l2cVoaXLvyU4aVgbNeWW1eJl+eHl4yIWNw1HnFL1VouYApIc0liGMg5KlpBge7O1JQyasWi3lDrlO3V2gXHpY7cq0kgPhUQeVjghjGbOdc/e975nQvW1LYsh14aPne+EPfiYFy9bc+cY0ii93F69WF2XiUd0AYd+Ckz8CoIeYNlhLxBN40V6MVYI8aFZ1uEdwOZAjMu6NF8ZHkmI9B5wiaGb2HCzJmfARhQH8R4Oh/9U5qKdzvmAF4wbRvNQa/THIwMs2OM2pY5HNFxk47avc64a1lfh4tFW2ksQEtfgQfxphbDw6JdXg0ewiN2wz9jn9TPX03xw5Y2LDsnHBbOCYfLnBNKipcUn6d4wcQb7Ho68pwSOq/MGeWOQGZQeXHinrj0SgTgFwXf21fUOuKFAQfigwIyRLUSdVATGfUp5kRxKlXHaG6jpvx9elUprYNnXLfd9WQB4lUtArjCCDW19GL1YtgtCbG/HGKl5PlNACt6f3ctHds2vJIzeLTp+8Z10bAxtGu0MtkWNHBMtf2Y9lmS2FdL7DmS+Y8g850L+LoDcx7QDH1TfM4qNSWd4cHEvdydz5K4JXGvhLh3wwjfWwYsU1nuHQX/XoJjD7Wy0DYFBdFQErUk6kcg6sjXQtx0l6Vq7urdGPGUJShbbZa77KkFS5WlfPYkdUvqXpS67xS4H+JIlKFtFX7KaLuwr1SX0exL2pa0fS9tc2UM3zFnKTvcRUcqmAeT9uO6L0jSlqR9P9u2+Kv4AGXYkWvM2MQLskQ+LcuyDLW34aeM2otm/su4TEtql9R+H7UfjH6nZrBzNaMYNv9OUcXLZFuG1DuaWnbCqhZOWFV5wipJ/fFInQegowH176H1ON8yxN69w32rYAO3lPuWJHZJ7PcRO5Kv41DnV8PhlHontc9yGZch9x78lJF7MSJZT5K7JPcHknscHNKfu4E9pesXxgwIF6NEQgdnnotvtHHsGy6z0SzuC3fghd6Ih54XbI6Btx/RYD6rhi6EocWhRn7F+reiquNQ8z+SpI2DjD+fGVxp5Bd6vYVNXgX8aU0jnEYT/0aMvmQC1mDg5wTIONWiQfBmXZ9cGs455HOuyXyGgMPYoZSYolpiB+h1mPJ4/O48bpMpE4NVoxrTro0v4qfKaEr+/HODfPpcU2ZzNqmKy8hSN98K3jFByxLgFXG574N44Ele7ADPmbk7l5SWgsrmrkXHtkutdF4O4GRWuUnL7nTmVNOZkirr2ce89dyzuJnccwHQ7LP07J3RIJm9TL5a8uU2vgu4lvKuzFbEUhXVkTJqD7uyd3Ei11R1AFJ1q1kqaqhF20XMmuW/F3P6uwi6mme73WFzkGa7aOCb6RrQt08d45pcWN6UpF61oB5guAYxJ7ZjVRjxAW+QG16NT4FrI/KAmDHo7bVrop31fPZ0HLlFh5aqNq2BSYdNqz0Yt8ZWp0tb5rDXGYzHo6ba63SGw/5CHDmZQMGIF628sSRTu4dZp7ojeHT0GvM8mmfUdcyp64t2emHWrOt4TTjUf8dYGGwjS+4758xHwBcYxtQ4p4ecZqq1+ySSr+uH1u7BTwmI2gUBpt2TAHrWAHoA6f23YImbDnizEFLht6dCVv8Of+OCVVje31giSyLrOSIr3kuYnoOB6fco7D1ZNRc6wOFPNRKK2LgTqH36TFD2zviNxZKzKBALnp7L64UKdOpoZMehuGOro1sVPkwqxQ+irBDpMUMkvy4W++MrEd5pap2ytbNTMLToyLVTIvwbQPi9CH5sRKnljpudgnlH3nFTIkoi6jkiKlkzuefv5mxGDf8YKjoK4Guo7GET71IjIw+oxnDDFYxeASXyTBo5pDAm66W4KaiOMbZeARpvboW2LXcPivuGjmEGREuQ7cJWxm61ls3zpVS8boary8fVeI3GHsYrtKIoSe9S63a64dTjzAOhPioXBLDhh1b3JOv8QOuVrvMFE/ieXOclV/oGuNKXWcUjw6nbgp8SOHULhm7dloSThNO3A6enBlIbfsqAVLCh67YlkCSQvhUgPbIA+tio7Nxxa3rhUDJ/a7pEpUTlc0Rlcr1teIJy5HhBUgp3s3gzdLSRy+xmxSPP3cGSGqner9LFD1GpSBO8eKnH3gv2Blq/THjtFfaCfSm8SnR/A+gWa+4Wkg9ezAIlobOxFdLpy4BOZ2ii9SoxBnqZVJo2F8IIPRsna9/f4Ifbk7V0mmYyBmkirnsm5R8UWQOk4bKvhJwil4VrpJMsnC2ks7xK537JgD29+v5mHBqc3b5s8Cep/qdmJX78shGP9LReopJ6bLlh0NaGZeZMg4I0P5SmTJKz/NWcBWUAdq8Yr/ELIXKyfPSfEBTKzBO5iG+7dlB4jos/08inz0B2uQOqQt6HyA+FzhHieGfiYLnC2Y9WIX8n1FFMx2Bs35jSWrEI508XhjOnCu9o6mA6n/k2193svuQBdpL3vRtNbXaaQ/ytlpmj8NQ8h+GZH8Bk+vcxGW7t+i3xl2Fb7Qz7Y2PQbraGqtkdDPu03RwMrDYFQJn9dofCs7G6FH9ZtPIH8JcCHJdgLP17GcuiHX8MS3bBVMa+ccb9GEJb9bfh95Lz6VXZGnfaHfxXdgBUjITfyYfCl7h5Prh5BBPjbwEzZijbhy4gSYE3jmeex6L+CL/Fq1IEtNziGONPLG3R19Ty1miQXdcFGhxBO+aEMn7bEyDCQQ1CMKG2H1LxJUytNw+Ewwe/KmqOtIv0zJRMhb9QOhP+IHPfx05NANfeeEwwwBm5nMAvTBUthldTwVrNF3xyTYNUbei7AdIIzMUE78eKBpB22ajFbiBlqbEjRKJhvIThlu0SVseBulqnU86BCn6VkFFyIMmBnhUHSnkJaeQ3eMnvjdnLzPpdTzmRoXGLSy+jjNVHwdRwMMB/JZiCp4Ud92AgMSUx9UwwlffpLF/Q+XKukTepVd1zI7i9hg0wLm45KTreE0cFcGV7ncsU+XgehhfV5ponfyaeiLlL5QJ+aRXWmVw+i0LAtrjNEThAxvsx76TIS9bwSpHDg4Nj/Zedf+lbB/vHO/88zuylIWvUVaxRlErlSN1fm68n01sEIgDwzoFmjNpDk/hcFnaHcJW8iPhR1Fz8AMQeg6DiEsUihwbAE4R8U+f8AxBLRtdc7BHpyB2oM46Kc60q7goTvlqFF1APpRViM0x8G+pIBeHwTtaiuaplrPSi/n1qfn48r85mX4Vfw9IIb5BWVFMMH2IGLPmy5MtPsNt69lzukaDbg1+DO6BbDIcBWSV0JXQfE7rLnVoUnbBfosH9Kq35svr55W9gkpB5PpD5z9HJLwUbWMWotTs+mhgzAI9Ye/iugUuW/LFytLv/bm9Hx+UINglVlbx8KfLsjt86xhlTdve3d/6pH/28+/b48dDW1QZlzhyDgiJt8BBnDok2ibanQxtuCzGSUwy4e7CWQdnRHoiDmPBoEOvDTxnECpEPBvKQWULseUKML2hHBZxl1rknB9ZAK9VYFxXWUl8tgfVMgRWtXW89n8MmWrx+jR48AY6G8FOGo4Ih1OAhLhoSRxJHT4cj13OPArxiHLNiOACihwc1+ACTbMcOrpX9g3396Hjzzd4OuUWjSV76sZA1bMJP2ZlqISTOsCmRJZH1LJElQnXAcnRp+FYiBALGqq4xpRqJAnFEHgavw2PWty4eoIYHgrwSLMsL1YkL05AUqSdQFYve24PD3zYPt3e2H03zMVTvuPe6cK6Wv/daYlNi8y/HZsrqAcnkNxgPz15FgZJxR4OacDcIjQ74mXgSZ9q9Tp/ub0GTAGih7c9GvU45HfMY2bVimOigCWUjd6Fq5aVlX7x6ibcIv6rU8nldnrdZTSVEvUJ/K2HQGEXaTvKEx+5uM29vmI6gI1q4CacE1vdoRMkY/DDaZXoQ0ZE+Z1HJuPFTHcMKcRmd1b7UclRtxkbATo1ihY6YOTbW0dTSq4OGxajP+buDJCOTjOwvZ2QFvvRoSOmWexQOC0cdD/IolDiROHk6YfxbXqgfDdhoalO6BJbY2cglUEL7a6H9/wGQz8mf'
PUBLIC_NOTICES = {'Automattic__mongoose__754c8a73e3a086408bac4ab3dc9bef0eb364f5dd.txt': '# MIT License\n\nCopyright (c) 2010-2013 LearnBoost <dev@learnboost.com>\nCopyright (c) 2013-2021 Automattic\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'Automattic__mongoose__a477fca5c4ea47711b7f1a354d1a7367442ef178.txt': '# MIT License\n\nCopyright (c) 2010-2013 LearnBoost <dev@learnboost.com>\nCopyright (c) 2013-2021 Automattic\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'Automattic__mongoose__b3b242a5027ff5f37695779de48d89a7b8c633e1.txt': '# MIT License\n\nCopyright (c) 2010-2013 LearnBoost <dev@learnboost.com>\nCopyright (c) 2013-2021 Automattic\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'django__django__1a9fba7a6c2a0acead19bbb45c25ceb0fdcca7c8.txt': 'Copyright (c) Django Software Foundation and individual contributors.\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without modification,\nare permitted provided that the following conditions are met:\n\n    1. Redistributions of source code must retain the above copyright notice,\n       this list of conditions and the following disclaimer.\n\n    2. Redistributions in binary form must reproduce the above copyright\n       notice, this list of conditions and the following disclaimer in the\n       documentation and/or other materials provided with the distribution.\n\n    3. Neither the name of Django nor the names of its contributors may be used\n       to endorse or promote products derived from this software without\n       specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES\n(INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON\nANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n(INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'django__django__80c997fc23b08897d8611b5439c1877485a4676f.txt': 'Copyright (c) Django Software Foundation and individual contributors.\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without modification,\nare permitted provided that the following conditions are met:\n\n    1. Redistributions of source code must retain the above copyright notice,\n       this list of conditions and the following disclaimer.\n\n    2. Redistributions in binary form must reproduce the above copyright\n       notice, this list of conditions and the following disclaimer in the\n       documentation and/or other materials provided with the distribution.\n\n    3. Neither the name of Django nor the names of its contributors may be used\n       to endorse or promote products derived from this software without\n       specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES\n(INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON\nANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n(INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'django__django__adeef108afa7c6c50959cf2c94a4d4efbf1fae91.txt': 'Copyright (c) Django Software Foundation and individual contributors.\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without modification,\nare permitted provided that the following conditions are met:\n\n    1. Redistributions of source code must retain the above copyright notice,\n       this list of conditions and the following disclaimer.\n\n    2. Redistributions in binary form must reproduce the above copyright\n       notice, this list of conditions and the following disclaimer in the\n       documentation and/or other materials provided with the distribution.\n\n    3. Neither the name of Django nor the names of its contributors may be used\n       to endorse or promote products derived from this software without\n       specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES\n(INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON\nANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n(INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'django__django__b33bab33709bbd74f4dfc376786ba0b3912c3f53.txt': 'Copyright (c) Django Software Foundation and individual contributors.\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without modification,\nare permitted provided that the following conditions are met:\n\n    1. Redistributions of source code must retain the above copyright notice,\n       this list of conditions and the following disclaimer.\n\n    2. Redistributions in binary form must reproduce the above copyright\n       notice, this list of conditions and the following disclaimer in the\n       documentation and/or other materials provided with the distribution.\n\n    3. Neither the name of Django nor the names of its contributors may be used\n       to endorse or promote products derived from this software without\n       specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT OWNER OR CONTRIBUTORS BE LIABLE FOR\nANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES\n(INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES;\nLOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON\nANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT\n(INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'encode__django-rest-framework__1af5b55758ac514c492ed3a5fc1a6fb244cdb257.txt': '# License\n\nCopyright © 2011-present, [Encode OSS Ltd](https://www.encode.io/).\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n* Redistributions of source code must retain the above copyright notice, this\n  list of conditions and the following disclaimer.\n\n* Redistributions in binary form must reproduce the above copyright notice,\n  this list of conditions and the following disclaimer in the documentation\n  and/or other materials provided with the distribution.\n\n* Neither the name of the copyright holder nor the names of its\n  contributors may be used to endorse or promote products derived from\n  this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'encode__django-rest-framework__41764f1f8cfb6d40b77b58c97cf9aeaab57c6b5c.txt': '# License\n\nCopyright © 2011-present, [Encode OSS Ltd](https://www.encode.io/).\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n* Redistributions of source code must retain the above copyright notice, this\n  list of conditions and the following disclaimer.\n\n* Redistributions in binary form must reproduce the above copyright notice,\n  this list of conditions and the following disclaimer in the documentation\n  and/or other materials provided with the distribution.\n\n* Neither the name of the copyright holder nor the names of its\n  contributors may be used to endorse or promote products derived from\n  this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'encode__django-rest-framework__7195f7e4c51527054623ea55fc4491786acc91f8.txt': '# License\n\nCopyright © 2011-present, [Encode OSS Ltd](https://www.encode.io/).\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n* Redistributions of source code must retain the above copyright notice, this\n  list of conditions and the following disclaimer.\n\n* Redistributions in binary form must reproduce the above copyright notice,\n  this list of conditions and the following disclaimer in the documentation\n  and/or other materials provided with the distribution.\n\n* Neither the name of the copyright holder nor the names of its\n  contributors may be used to endorse or promote products derived from\n  this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'encode__django-rest-framework__b92edf53e21b2de16af5d13529591f1c0d415f93.txt': '# License\n\nCopyright © 2011-present, [Encode OSS Ltd](https://www.encode.io/).\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n* Redistributions of source code must retain the above copyright notice, this\n  list of conditions and the following disclaimer.\n\n* Redistributions in binary form must reproduce the above copyright notice,\n  this list of conditions and the following disclaimer in the documentation\n  and/or other materials provided with the distribution.\n\n* Neither the name of the copyright holder nor the names of its\n  contributors may be used to endorse or promote products derived from\n  this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS" AND\nANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED\nWARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'expressjs__express__023767fe9872e029271df1418f73401bff20ff40.txt': "(The MIT License)\n\nCopyright (c) 2009-2014 TJ Holowaychuk <tj@vision-media.ca>\nCopyright (c) 2013-2014 Roman Shtylman <shtylman+expressjs@gmail.com>\nCopyright (c) 2014-2015 Douglas Christopher Wilson <doug@somethingdoug.com>\n\nPermission is hereby granted, free of charge, to any person obtaining\na copy of this software and associated documentation files (the\n'Software'), to deal in the Software without restriction, including\nwithout limitation the rights to use, copy, modify, merge, publish,\ndistribute, sublicense, and/or sell copies of the Software, and to\npermit persons to whom the Software is furnished to do so, subject to\nthe following conditions:\n\nThe above copyright notice and this permission notice shall be\nincluded in all copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED 'AS IS', WITHOUT WARRANTY OF ANY KIND,\nEXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.\nIN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY\nCLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,\nTORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE\nSOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n", 'expressjs__express__3ce6d0eb86e9d93529ff3191c6bb5db8ce6e72c8.txt': "(The MIT License)\n\nCopyright (c) 2009-2014 TJ Holowaychuk <tj@vision-media.ca>\nCopyright (c) 2013-2014 Roman Shtylman <shtylman+expressjs@gmail.com>\nCopyright (c) 2014-2015 Douglas Christopher Wilson <doug@somethingdoug.com>\n\nPermission is hereby granted, free of charge, to any person obtaining\na copy of this software and associated documentation files (the\n'Software'), to deal in the Software without restriction, including\nwithout limitation the rights to use, copy, modify, merge, publish,\ndistribute, sublicense, and/or sell copies of the Software, and to\npermit persons to whom the Software is furnished to do so, subject to\nthe following conditions:\n\nThe above copyright notice and this permission notice shall be\nincluded in all copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED 'AS IS', WITHOUT WARRANTY OF ANY KIND,\nEXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.\nIN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY\nCLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,\nTORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE\nSOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n", 'expressjs__express__41386506a9ca6874a79b304f1baafd50522e6078.txt': "(The MIT License)\n\nCopyright (c) 2009-2014 TJ Holowaychuk <tj@vision-media.ca>\nCopyright (c) 2013-2014 Roman Shtylman <shtylman+expressjs@gmail.com>\nCopyright (c) 2014-2015 Douglas Christopher Wilson <doug@somethingdoug.com>\n\nPermission is hereby granted, free of charge, to any person obtaining\na copy of this software and associated documentation files (the\n'Software'), to deal in the Software without restriction, including\nwithout limitation the rights to use, copy, modify, merge, publish,\ndistribute, sublicense, and/or sell copies of the Software, and to\npermit persons to whom the Software is furnished to do so, subject to\nthe following conditions:\n\nThe above copyright notice and this permission notice shall be\nincluded in all copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED 'AS IS', WITHOUT WARRANTY OF ANY KIND,\nEXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.\nIN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY\nCLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,\nTORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE\nSOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.\n", 'fastapi__fastapi__0ab3e037a6bd406ae8462841864b5c6337c8b329.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018 Sebastián Ramírez\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'fastapi__fastapi__129c9cfcd33c3ed4371c9674e724fd838a292e2b.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018 Sebastián Ramírez\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'fastapi__fastapi__9528746e3493a1cdd17da2eff97c48ef65290713.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018 Sebastián Ramírez\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'fastapi__fastapi__bb7f314287bf36e04539c572a2256d94ea45f0d6.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018 Sebastián Ramírez\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'pallets__flask__3596b1ab61cea85edb8970e83ff61daa073facf8.txt': 'Copyright 2010 Pallets\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are\nmet:\n\n1.  Redistributions of source code must retain the above copyright\n    notice, this list of conditions and the following disclaimer.\n\n2.  Redistributions in binary form must reproduce the above copyright\n    notice, this list of conditions and the following disclaimer in the\n    documentation and/or other materials provided with the distribution.\n\n3.  Neither the name of the copyright holder nor the names of its\n    contributors may be used to endorse or promote products derived from\n    this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS\n"AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT\nLIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A\nPARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT\nHOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL,\nSPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED\nTO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR\nPROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF\nLIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING\nNEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'pallets__flask__d73fa1cdcbd8b1465c151db8924ba58b1dd14e35.txt': 'Copyright 2010 Pallets\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are\nmet:\n\n1.  Redistributions of source code must retain the above copyright\n    notice, this list of conditions and the following disclaimer.\n\n2.  Redistributions in binary form must reproduce the above copyright\n    notice, this list of conditions and the following disclaimer in the\n    documentation and/or other materials provided with the distribution.\n\n3.  Neither the name of the copyright holder nor the names of its\n    contributors may be used to endorse or promote products derived from\n    this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS\n"AS IS" AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT\nLIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A\nPARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT\nHOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL,\nSPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED\nTO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR\nPROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF\nLIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING\nNEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS\nSOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'pydantic__pydantic__03de65fc2f44faa4b32e05ce0a5d499cdf38f89b.txt': 'The MIT License (MIT)\n\nCopyright (c) 2017 to present Pydantic Services Inc. and individual contributors.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'pydantic__pydantic__8cd3bec6c7e1d360d92193841613465d30f63955.txt': 'The MIT License (MIT)\n\nCopyright (c) 2017 to present Pydantic Services Inc. and individual contributors.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'pydantic__pydantic__e0fb2cead13b1f604505ecb3621c5d6781e9307a.txt': 'The MIT License (MIT)\n\nCopyright (c) 2017 to present Pydantic Services Inc. and individual contributors.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'pydantic__pydantic__e1ca33220585e8f745000eff4b067e04fe7d83e9.txt': 'The MIT License (MIT)\n\nCopyright (c) 2017 to present Pydantic Services Inc. and individual contributors.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sequelize__sequelize__7a976d561215ccbe1dd420e54c36a63b1794ee14.txt': 'MIT License\n\nCopyright (c) 2014-present Sequelize Authors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sequelize__sequelize__960de1ecd084dddb47ca6d99c320ba21c6445931.txt': 'MIT License\n\nCopyright (c) 2014-present Sequelize Authors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sequelize__sequelize__9c3ec5c775d43867184f6289bb891648098e9836.txt': 'MIT License\n\nCopyright (c) 2014-present Sequelize contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sequelize__sequelize__c8ba82b28d1ecf181ee889a8d53dec43949f69d9.txt': 'MIT License\n\nCopyright (c) 2014-present Sequelize contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sqlalchemy__sqlalchemy__632c664fe9ff67ebaecc6ceed99262afdf80011f.txt': 'Copyright 2005-2026 SQLAlchemy authors and contributors <see AUTHORS file>.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of\nthis software and associated documentation files (the "Software"), to deal in\nthe Software without restriction, including without limitation the rights to\nuse, copy, modify, merge, publish, distribute, sublicense, and/or sell copies\nof the Software, and to permit persons to whom the Software is furnished to do\nso, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'sqlalchemy__sqlalchemy__f46ead8007f8f814f0639b785c637b29afdaec79.txt': 'Copyright 2005-2026 SQLAlchemy authors and contributors <see AUTHORS file>.\n\nPermission is hereby granted, free of charge, to any person obtaining a copy of\nthis software and associated documentation files (the "Software"), to deal in\nthe Software without restriction, including without limitation the rights to\nuse, copy, modify, merge, publish, distribute, sublicense, and/or sell copies\nof the Software, and to permit persons to whom the Software is furnished to do\nso, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'vuejs__core__1f867d9c6bdd7666b9b21a1488a0d79d9fa9a1d6.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018-present, Yuxi (Evan) You and Vue contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'vuejs__core__2e9d110d8ce90d38f2fd45e2c9648ffb01644997.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018-present, Yuxi (Evan) You and Vue contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'vuejs__core__931497fa830291c5897e3088d3ed38c734e589f1.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018-present, Yuxi (Evan) You and Vue contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'vuejs__core__e291103616de6ad7984f57bbb72cdaed3c5249a0.txt': 'The MIT License (MIT)\n\nCopyright (c) 2018-present, Yuxi (Evan) You and Vue contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in\nall copies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN\nTHE SOFTWARE.\n', 'manifest.json': '{\n  "label_policy": "weak_patch_overlap_v1",\n  "repositories": [\n    {\n      "repo": "pallets/flask",\n      "task": 6168,\n      "base_commit": "d73fa1cdcbd8b1465c151db8924ba58b1dd14e35",\n      "license_file": "pallets__flask__d73fa1cdcbd8b1465c151db8924ba58b1dd14e35.txt",\n      "license_path": "LICENSE.txt",\n      "candidates": 14\n    },\n    {\n      "repo": "django/django",\n      "task": 22144,\n      "base_commit": "adeef108afa7c6c50959cf2c94a4d4efbf1fae91",\n      "license_file": "django__django__adeef108afa7c6c50959cf2c94a4d4efbf1fae91.txt",\n      "license_path": "LICENSE",\n      "candidates": 11\n    },\n    {\n      "repo": "encode/django-rest-framework",\n      "task": 10066,\n      "base_commit": "7195f7e4c51527054623ea55fc4491786acc91f8",\n      "license_file": "encode__django-rest-framework__7195f7e4c51527054623ea55fc4491786acc91f8.txt",\n      "license_path": "LICENSE.md",\n      "candidates": 9\n    },\n    {\n      "repo": "sequelize/sequelize",\n      "task": 18536,\n      "base_commit": "9c3ec5c775d43867184f6289bb891648098e9836",\n      "license_file": "sequelize__sequelize__9c3ec5c775d43867184f6289bb891648098e9836.txt",\n      "license_path": "LICENSE",\n      "candidates": 13\n    },\n    {\n      "repo": "sequelize/sequelize",\n      "task": 18530,\n      "base_commit": "7a976d561215ccbe1dd420e54c36a63b1794ee14",\n      "license_file": "sequelize__sequelize__7a976d561215ccbe1dd420e54c36a63b1794ee14.txt",\n      "license_path": "LICENSE",\n      "candidates": 14\n    },\n    {\n      "repo": "Automattic/mongoose",\n      "task": 16592,\n      "base_commit": "a477fca5c4ea47711b7f1a354d1a7367442ef178",\n      "license_file": "Automattic__mongoose__a477fca5c4ea47711b7f1a354d1a7367442ef178.txt",\n      "license_path": "LICENSE.md",\n      "candidates": 12\n    },\n    {\n      "repo": "Automattic/mongoose",\n      "task": 16585,\n      "base_commit": "754c8a73e3a086408bac4ab3dc9bef0eb364f5dd",\n      "license_file": "Automattic__mongoose__754c8a73e3a086408bac4ab3dc9bef0eb364f5dd.txt",\n      "license_path": "LICENSE.md",\n      "candidates": 13\n    },\n    {\n      "repo": "vuejs/core",\n      "task": 15908,\n      "base_commit": "2e9d110d8ce90d38f2fd45e2c9648ffb01644997",\n      "license_file": "vuejs__core__2e9d110d8ce90d38f2fd45e2c9648ffb01644997.txt",\n      "license_path": "LICENSE",\n      "candidates": 11\n    },\n    {\n      "repo": "vuejs/core",\n      "task": 15907,\n      "base_commit": "931497fa830291c5897e3088d3ed38c734e589f1",\n      "license_file": "vuejs__core__931497fa830291c5897e3088d3ed38c734e589f1.txt",\n      "license_path": "LICENSE",\n      "candidates": 16\n    }\n  ],\n  "skipped": [\n    {\n      "repo": "pallets/flask",\n      "task": 6133,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "fastapi/fastapi",\n      "task": 16520,\n      "reason": "Row 1: possible private data or credentials. Sanitize locally before uploading; this scan is not comprehensive."\n    },\n    {\n      "repo": "fastapi/fastapi",\n      "task": 16518,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "fastapi/fastapi",\n      "task": 16517,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "fastapi/fastapi",\n      "task": 16512,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "django/django",\n      "task": 22149,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "django/django",\n      "task": 22141,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "django/django",\n      "task": 22139,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "encode/django-rest-framework",\n      "task": 10071,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "encode/django-rest-framework",\n      "task": 10056,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "encode/django-rest-framework",\n      "task": 10052,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "sqlalchemy/sqlalchemy",\n      "task": 13669,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "sqlalchemy/sqlalchemy",\n      "task": 13662,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "pydantic/pydantic",\n      "task": 14008,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "pydantic/pydantic",\n      "task": 14006,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "pydantic/pydantic",\n      "task": 13997,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "pydantic/pydantic",\n      "task": 13996,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "expressjs/express",\n      "task": 7468,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "expressjs/express",\n      "task": 7466,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "expressjs/express",\n      "task": 7459,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "sequelize/sequelize",\n      "task": 18533,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "Automattic/mongoose",\n      "task": 16590,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "Automattic/mongoose",\n      "task": 16588,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "vuejs/core",\n      "task": 15913,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    },\n    {\n      "repo": "vuejs/core",\n      "task": 15911,\n      "reason": "Insufficient complete small changed/unchanged units; no truncation or invented labels"\n    }\n  ]\n}'}

if USE_DEMO_DATA:
    rows = make_demo_rows()
    print("DEMO ONLY: original synthetic labels; not proof of production retrieval quality.")
elif USE_PUBLIC_DATA:
    import base64, zlib
    if not BUNDLED_PUBLIC_DATA:
        raise ValueError("No bundled pilot: regenerate the notebook after collecting public data.")
    rows = [json.loads(line) for line in zlib.decompress(base64.b64decode(BUNDLED_PUBLIC_DATA)).decode().splitlines() if line.strip()]
    print("PUBLIC PILOT: patch-overlap weak labels, not verified complete context. No production promotion.")
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one reviewed JSONL file.")
    raw = next(iter(uploaded.values()))
    if len(raw) > 100 * 1024 * 1024:
        raise ValueError("Dataset exceeds this notebook's 100 MB input limit.")
    rows = [json.loads(line) for line in raw.decode("utf-8-sig").splitlines() if line.strip()]
    del uploaded, raw
rows = validate_rows(rows)
weak_labels = any(r.get("labels_verified") is False for r in rows)
splits = split_repositories(rows, SEED)
fingerprint = dataset_hash(rows)
run_config = dict(base_model=BASE_MODEL, revision=BASE_REVISION, seed=SEED, epochs=EPOCHS,
                  batch_size=BATCH_SIZE, gradient_accumulation=GRAD_ACCUMULATION,
                  learning_rate=LEARNING_RATE, max_length=MAX_LENGTH, max_query_tokens=MAX_QUERY_TOKENS,
                  evaluation_budget=EVALUATION_BUDGET, demo=USE_DEMO_DATA, weak_labels=weak_labels, dataset_sha256=fingerprint,
                  torch_version=str(torch.__version__), transformers_version=importlib.metadata.version("transformers"))
run_id = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR = WORK_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)
if USE_PUBLIC_DATA and not USE_DEMO_DATA:
    notice_directory = RUN_DIR / "public_source_notices"
    notice_directory.mkdir(exist_ok=True)
    for name, notice in PUBLIC_NOTICES.items():
        (notice_directory / name).write_text(notice, encoding="utf-8")
split_manifest = {}
repo_sets = {}
for name, subset in splits.items():
    ids = {r["repo_id"] for r in subset}
    repo_sets[name] = ids
    split_manifest[name] = dict(repositories=len(ids), tasks=len({(r["repo_id"],r["task_id"]) for r in subset}), candidates=len(subset),
        repository_hashes=sorted(hashlib.sha256(repo.encode()).hexdigest() for repo in ids))
assert not repo_sets["train"] & repo_sets["validation"]
assert not repo_sets["train"] & repo_sets["test"]
assert not repo_sets["validation"] & repo_sets["test"]
overlap = set(hashlib.sha256(passage_text(r).encode()).hexdigest() for r in splits["train"]) & set(hashlib.sha256(passage_text(r).encode()).hexdigest() for r in splits["test"])
if overlap:
    raise ValueError("Identical candidate text crosses train/test repositories. Deduplicate independent projects before evaluating.")
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
(RUN_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, indent=2), encoding="utf-8")
print(json.dumps(split_manifest, indent=2))
print("Run ID:", run_id)


## 5. Load the pinned pretrained model and validate sequence lengths
The base model is Apache-2.0 licensed. Downloads are public and do not need an account token. This is a general passage reranker; its suitability for code must be measured rather than assumed.

Training rejects overlong task/unit pairs instead of silently discarding a model relation or function tail. Split a long file into complete, independently labeled code units, shorten task text, or increase `MAX_LENGTH` up to 512. Exported inference can score long candidates in windows while preserving their original content.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import torch.nn.functional as F
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, revision=BASE_REVISION,
            use_safetensors=True, trust_remote_code=False).to(DEVICE)
assert model.config.num_labels == 1, "This pipeline requires a one-logit reranker."
assert 32 <= MAX_LENGTH <= min(512, model.config.max_position_embeddings)
long_units = 0
for row in rows:
    query_ids = tokenizer.encode(query_text(row), add_special_tokens=False)
    if len(query_ids) > MAX_QUERY_TOKENS:
        raise ValueError("A task exceeds MAX_QUERY_TOKENS; shorten its requirements or deliberately raise the query limit.")
    size = len(tokenizer.encode(query_text(row), passage_text(row), add_special_tokens=True, truncation=False))
    long_units += size > MAX_LENGTH
if long_units:
    raise ValueError(f"{long_units} training units exceed MAX_LENGTH. Split them into complete functions/model blocks and relabel; no automatic training truncation was applied.")
print("All training units fit without truncation. Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


## 6. Shared inference function
This scores candidates. It does not modify source or decide which dependencies are mandatory. Scores are ranking signals, not calibrated probabilities.


In [ ]:
@torch.inference_mode()
def neural_scores(query, candidates, active_model=None):
    active_model = active_model if active_model is not None else model
    active_model.eval()
    result = []
    for start in range(0, len(candidates), BATCH_SIZE):
        batch = candidates[start:start + BATCH_SIZE]
        encoded_batch = tokenizer([query] * len(batch), [passage_text(r) for r in batch],
                                  padding=True, truncation=False, return_tensors="pt").to(DEVICE)
        logits = active_model(**encoded_batch).logits.view(-1).float().cpu().tolist()
        result.extend(logits)
    return result

baseline_validation = evaluate(splits["validation"], neural_scores, EVALUATION_BUDGET)
print("Pretrained validation metrics:", json.dumps(baseline_validation, indent=2))
print("Token cost = compact JSON UTF-8 bytes / 3, an estimate; not a provider tokenizer.")


## 7. Fine-tune using graded pairwise relevance
The loss teaches `essential > useful > irrelevant` for the **same task**. Epoch selection uses validation repositories only; the held-out test results are not used to select checkpoints.

The notebook saves an optimizer checkpoint after every epoch. With Drive persistence enabled, rerunning the same config/data resumes at the next epoch. An interruption during an epoch loses that epoch's unfinished updates. This notebook only reloads its own checkpoint path; never substitute an untrusted `.pt` file.


In [ ]:
pairs = training_pairs(splits["train"], SEED)
if not pairs:
    raise ValueError("No graded positive/negative training pairs were formed.")
if len(pairs) > 50000:
    raise ValueError("Start with fewer than 50,000 training pairs for this Colab experiment.")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=.01)
use_fp16 = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
last_path = RUN_DIR / "last.pt"
start_epoch, best_key, best_directory, history = 0, (-1., -1., -1.), None, []
if RESUME_IF_AVAILABLE and last_path.exists():
    checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=True)
    if checkpoint["run_id"] != run_id:
        raise ValueError("Checkpoint/config mismatch; start a separate run.")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])
    start_epoch = checkpoint["completed_epochs"]
    best_key, best_directory, history = tuple(checkpoint["best_key"]), Path(checkpoint["best_directory"]), checkpoint["history"]
    torch.set_rng_state(checkpoint["torch_rng"].cpu())
    if DEVICE.type == "cuda" and checkpoint["cuda_rng"]:
        torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng"]])
    del checkpoint
    print("Resuming after completed epoch", start_epoch)

print("Training pairs:", len(pairs), "Epochs:", EPOCHS, "Effective pair batch:", BATCH_SIZE * GRAD_ACCUMULATION)
training_started = time.perf_counter()
for epoch in range(start_epoch, EPOCHS):
    ordered = list(pairs)
    random.Random(SEED + epoch).shuffle(ordered)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    batches = math.ceil(len(ordered) / BATCH_SIZE)
    epoch_losses = []
    progress = tqdm(range(batches), desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for batch_index in progress:
        batch = ordered[batch_index * BATCH_SIZE:(batch_index + 1) * BATCH_SIZE]
        queries = [pair[0] for pair in batch]
        positives, negatives = [pair[1] for pair in batch], [pair[2] for pair in batch]
        inputs = tokenizer(queries + queries, positives + negatives, padding=True, truncation=False, return_tensors="pt").to(DEVICE)
        weights = torch.tensor([pair[3] for pair in batch], dtype=torch.float32, device=DEVICE)
        group_size = min(GRAD_ACCUMULATION, batches - (batch_index // GRAD_ACCUMULATION) * GRAD_ACCUMULATION)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_fp16):
            logits = model(**inputs).logits.view(-1)
            positive_logits, negative_logits = logits[:len(batch)], logits[len(batch):]
            raw_loss = (F.softplus(-(positive_logits - negative_logits).float()) * weights).sum() / weights.sum()
            loss = raw_loss / group_size
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite loss; stop and inspect data or lower the learning rate.")
        scaler.scale(loss).backward()
        if (batch_index + 1) % GRAD_ACCUMULATION == 0 or batch_index + 1 == batches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        epoch_losses.append(float(raw_loss.detach().cpu()))
        progress.set_postfix(loss=round(sum(epoch_losses) / len(epoch_losses), 4))
    metrics = evaluate(splits["validation"], neural_scores, EVALUATION_BUDGET)
    key = (metrics["ndcg_at_5"], metrics["essential_recall_at_budget"], -metrics["irrelevant_token_fraction"])
    history.append(dict(epoch=epoch + 1, loss=sum(epoch_losses) / len(epoch_losses), validation=metrics))
    print("Validation:", json.dumps(metrics, indent=2))
    if key > best_key:
        best_key = key
        best_directory = RUN_DIR / f"best_epoch_{epoch + 1}"
        model.save_pretrained(best_directory, safe_serialization=True)
        tokenizer.save_pretrained(best_directory)
    state = dict(run_id=run_id, completed_epochs=epoch + 1, model=model.state_dict(), optimizer=optimizer.state_dict(),
                 scaler=scaler.state_dict(), best_key=list(best_key), best_directory=str(best_directory), history=history,
                 torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [])
    temporary = RUN_DIR / "last.tmp"
    torch.save(state, temporary)
    os.replace(temporary, last_path)
    del state
    (RUN_DIR / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
print("Training time this session, minutes:", round((time.perf_counter() - training_started) / 60, 2))
if best_directory is None or not (best_directory / "model.safetensors").exists():
    raise RuntimeError("No complete best checkpoint exists. Inspect training output before exporting.")


## 8. Evaluate the selected model on untouched test repositories
This compares three rankers on identical candidate sets and budgets: a keyword baseline, the original pretrained model, and the selected fine-tuned checkpoint. It measures **candidate ranking**, not full-repository retrieval recall or code compilation. A required file missing from the candidate set cannot be recovered by a reranker.

`essential_recall_at_budget` should increase; `irrelevant_token_fraction` should decrease. Tiny demo test sets and synthetic labels are not statistically reliable. Freeze a real test set before tuning, and do not repeatedly tune against it.


In [ ]:
del optimizer
del model
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(str(best_directory), use_safetensors=True, trust_remote_code=False).to(DEVICE)
trained_validation = evaluate(splits["validation"], neural_scores, EVALUATION_BUDGET)
trained_test = evaluate(splits["test"], neural_scores, EVALUATION_BUDGET)
del model
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, revision=BASE_REVISION,
            use_safetensors=True, trust_remote_code=False).to(DEVICE)
pretrained_test = evaluate(splits["test"], neural_scores, EVALUATION_BUDGET)
keyword_test = evaluate(splits["test"], keyword_scores, EVALUATION_BUDGET)
del model
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

beats_pretrained = (trained_test["ndcg_at_5"] >= pretrained_test["ndcg_at_5"] + .01
    and trained_test["essential_recall_at_budget"] >= pretrained_test["essential_recall_at_budget"]
    and trained_test["irrelevant_token_fraction"] <= pretrained_test["irrelevant_token_fraction"])
evaluation_report = dict(format="contexa.reranker-evaluation.v1", demo=USE_DEMO_DATA,
    production_recommended=False, candidate_quality_gate_passed=bool(beats_pretrained and not USE_DEMO_DATA and not weak_labels),
    weak_labels=weak_labels, metric_interpretation="With weak labels, essential recall measures observed changed-unit coverage; irrelevant fraction means unchanged-unit token fraction, not verified irrelevance.",
    reason="Demo data cannot establish production quality." if USE_DEMO_DATA else ("Patch-derived weak labels cannot establish complete-context correctness; production promotion is disabled." if weak_labels else "Human review, broader held-out evaluation, latency and end-to-end generated-code checks are still required."),
    run_id=run_id, config=run_config, splits=split_manifest, pretrained_validation=baseline_validation,
    trained_validation=trained_validation, test={"keyword":keyword_test,"pretrained":pretrained_test,"trained":trained_test},
    scope="Ranking within supplied candidates; no claim of complete-repo recall or first-try compilation.",
    token_estimate="UTF-8 bytes / 3; no provider-token or message-limit guarantee.")
(RUN_DIR / "evaluation_report.json").write_text(json.dumps(evaluation_report, indent=2), encoding="utf-8")
print(json.dumps(evaluation_report["test"], indent=2))
print(evaluation_report["reason"])
print("Candidate quality gate:", evaluation_report["candidate_quality_gate_passed"], "Production approved:", False)


## 9. Export the model and a CPU-compatible inference adapter
The ZIP includes safe-tensor model weights, tokenizer, provenance/configuration, evaluation report and `inference.py`. It excludes raw training data and resume checkpoints. Treat the weights as a candidate artifact; don't automatically replace Contexa's current selector.

The adapter scores long source in bounded overlapping windows and returns the maximum score per unit. Selection keeps whole original units. It deliberately leaves mandatory dependency expansion, secrecy checks and final pack budgets to Contexa. Long inputs beyond the configured window cap raise an error rather than silently omit the tail.


In [ ]:
INFERENCE_SOURCE = '"""CPU/GPU ranking adapter. Loads safetensors only; never executes candidate code."""\nimport math\nfrom pathlib import Path\nimport torch\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n\nclass ContexaReranker:\n    def __init__(self, model_dir, device="cpu", max_length=384, max_query_tokens=128, batch_size=4, max_windows_per_unit=32):\n        path = Path(model_dir)\n        self.device = torch.device(device)\n        self.tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True, trust_remote_code=False)\n        self.model = AutoModelForSequenceClassification.from_pretrained(path, local_files_only=True, trust_remote_code=False, use_safetensors=True).to(self.device).eval()\n        if self.model.config.num_labels != 1:\n            raise ValueError("Expected a one-logit relevance model.")\n        self.max_length, self.max_query_tokens = max_length, max_query_tokens\n        self.batch_size, self.max_windows = batch_size, max_windows_per_unit\n        if not 32 <= max_length <= min(512, self.model.config.max_position_embeddings):\n            raise ValueError("Unsupported max_length.")\n        if batch_size < 1 or max_windows_per_unit < 1:\n            raise ValueError("Batch/window limits must be positive.")\n\n    @torch.inference_mode()\n    def score(self, query, units):\n        if not isinstance(query, str) or not query.strip():\n            raise ValueError("A nonempty task query is required.")\n        if len(units) > 256:\n            raise ValueError("Retrieve at most 256 candidate units before reranking.")\n        query_ids = self.tokenizer.encode(query, add_special_tokens=False)\n        if len(query_ids) > self.max_query_tokens:\n            raise ValueError("Task exceeds the query limit; shorten it rather than silently truncating requirements.")\n        available = self.max_length - len(query_ids) - self.tokenizer.num_special_tokens_to_add(pair=True)\n        if available < 32:\n            raise ValueError("Insufficient token space for source context.")\n        scores = [-math.inf] * len(units)\n        batch, owners = [], []\n\n        def flush():\n            if not batch:\n                return\n            tokens = self.tokenizer.pad(batch, padding=True, return_tensors="pt").to(self.device)\n            values = self.model(**tokens).logits.view(-1).float().cpu().tolist()\n            for owner, value in zip(owners, values):\n                if not math.isfinite(value):\n                    raise RuntimeError("Model returned a nonfinite score.")\n                scores[owner] = max(scores[owner], value)\n            batch.clear()\n            owners.clear()\n\n        for owner, unit in enumerate(units):\n            if not isinstance(unit.get("content"), str) or not unit["content"].strip():\n                raise ValueError("Every unit needs complete nonempty source content.")\n            if len(unit["content"].encode("utf-8")) > 128_000:\n                raise ValueError("A source unit exceeds 128 KB; split it into complete functions or model blocks.")\n            passage = f"path: {unit.get(\'path\', \'\')}\\nrole: {unit.get(\'role\', \'source\')}\\ncode:\\n{unit[\'content\']}"\n            ids = self.tokenizer.encode(passage, add_special_tokens=False)\n            step, start, windows = max(1, available - 64), 0, 0\n            while start < len(ids):\n                windows += 1\n                if windows > self.max_windows:\n                    raise ValueError("A candidate exceeds the window cap; split it into complete functions/types. No tail was silently ignored.")\n                piece = ids[start:start + available]\n                batch.append(self.tokenizer.prepare_for_model(query_ids, piece, add_special_tokens=True, truncation=False, return_attention_mask=True))\n                owners.append(owner)\n                if len(batch) >= self.batch_size:\n                    flush()\n                if start + available >= len(ids):\n                    break\n                start += step\n        flush()\n        return scores\n'

bundle = RUN_DIR / "export"
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(best_directory, bundle / "model")
if (RUN_DIR / "public_source_notices").exists():
    shutil.copytree(RUN_DIR / "public_source_notices", bundle / "public_source_notices")
(bundle / "inference.py").write_text(INFERENCE_SOURCE, encoding="utf-8")
(bundle / "requirements.txt").write_text("torch>=2.6,<3\ntransformers==4.57.6\nsafetensors==0.7.0\n", encoding="utf-8")
for name in ("evaluation_report.json", "run_config.json", "split_manifest.json", "training_history.json"):
    shutil.copy2(RUN_DIR / name, bundle / name)
card = f"""# Contexa context reranker — candidate model

Base: {BASE_MODEL} at {BASE_REVISION}; base model license Apache-2.0.
Training data mode: {'synthetic demonstration' if USE_DEMO_DATA else 'read run_config.json and evaluation_report.json for label provenance'}.
Dataset SHA256: {fingerprint}. Run ID: {run_id}.
Model selection: repository-disjoint validation; final test uses held-out repositories.
Production approved: NO. Read evaluation_report.json before integrating.
Scores are not calibrated probabilities. This ranks provided candidates, not an entire repository.
Keep Contexa's credential filters, dependency rules and estimated-budget checks.
No guarantee of exact context, compile success, or privacy of training data encoded in weights.
Training-data licensing/permission must be checked before distributing a custom-data model.
See the base model card for upstream notices: https://huggingface.co/{BASE_MODEL}
"""
(bundle / "MODEL_CARD.md").write_text(card, encoding="utf-8")
readme = """# Using the exported candidate
Install requirements in a separate Python environment with enough RAM.
Do not deserialize untrusted .pt files; inference loads safetensors only.

from inference import ContexaReranker
ranker = ContexaReranker('model', device='cpu')
units = [{'path':'src/models/Interview.ts','role':'model','content':'export type Interview = {id:number};'}]
scores = ranker.score('Add an interview endpoint for recruiters', units)

The list of scores matches the input unit order. Preserve original source when building packs.
Run latency/memory benchmarks and inspect the evaluation report before enabling this in Contexa.
"""
(bundle / "README.md").write_text(readme, encoding="utf-8")
archive_path = Path(shutil.make_archive(str(RUN_DIR / ("contexa-reranker-demo" if USE_DEMO_DATA else "contexa-reranker-candidate")), "zip", bundle))
archive_sha = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print("ZIP:", archive_path.name, "Size MB:", round(archive_path.stat().st_size / 2**20, 1))
print("SHA256:", archive_sha)


## 10. Reload the exported adapter and smoke-test it
This checks the packaged model can be loaded through the same CPU interface intended for local integration.


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("contexa_export_inference", bundle / "inference.py")
export_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(export_module)
ranker = export_module.ContexaReranker(bundle / "model", device="cpu", max_length=MAX_LENGTH, max_query_tokens=MAX_QUERY_TOKENS)
sample_group = [r for r in splits["test"] if r["task_id"] == splits["test"][0]["task_id"]]
scores = ranker.score(query_text(sample_group[0]), sample_group)
assert len(scores) == len(sample_group) and all(math.isfinite(score) for score in scores)
print("CPU export reload passed for", len(scores), "candidate units.")
del ranker
gc.collect()


## 11. Download your artifacts
Send back **`evaluation_report.json` first** so we can review whether training improved anything. Keep the ZIP for integration. A demo-trained ZIP is for testing the integration, not a customer-quality release.

If the browser blocks multiple automatic downloads, run the two download calls separately. If Drive persistence is on, the files are also in your `MyDrive/ContexaTraining/<run-id>/` folder.


In [ ]:
from google.colab import files
files.download(str(RUN_DIR / "evaluation_report.json"))
files.download(str(archive_path))


## Next steps and references
- Use real, sanitized, developer-reviewed labels across repositories and stacks, including hard negatives and tasks where the right action is to ask for missing information.
- This notebook only compares a keyword baseline with pretrained/trained rerankers. Before deployment, also compare the full current Contexa pipeline, candidate-retrieval recall, dependency completeness, latency/RAM and generated-code compile/test outcomes.
- Integration should retrieve a bounded candidate set, rerank it, add mandatory imports/models/enums, then budget whole units. Keep a fallback if the model is missing, slow or worse than the existing selector.
- [Sentence Transformers training guidance](https://www.sbert.net/docs/cross_encoder/training_overview.html), [base model card](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2), [Transformers training](https://huggingface.co/docs/transformers/training), [Colab runtime limitations](https://research.google.com/colaboratory/faq.html).

**Verification note:** this notebook's Python syntax, demo data validation, split separation and metric/budget helpers were tested locally without installing ML packages. GPU training and model download/reload require your Colab run and have not been claimed as completed.
